# COMP5318 Assignment 1: Rice Classification

##### Group number: 24
##### Student 1 SID: 560072222
##### Student 2 SID: ...  
##### Student 3 SID: ... 


## **1. Data Pre-processing**

In [1]:
# Import all libraries
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold

In [2]:
# Ignore future warnings
from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)

In [3]:
# Load the rice dataset

def load_dataset(file_path):
    """Load a dataset whose class label is the last column, treating '?' as missing.

    Arguments:
        file_path: path to a csv file with a header row and the class in the last column
    Returns:
        a pandas DataFrame with missing values represented as NaN
    """
    return pd.read_csv(file_path, na_values='?')


rice_data = load_dataset('rice-final2.csv')
print("Loaded {} examples with {} features.".format(rice_data.shape[0], rice_data.shape[1] - 1))
print(rice_data.head(10))

Loaded 1400 examples with 7 features.
      Area   Perimiter  Major_Axis_Length  Minor_Axis_Length  Eccentricity  \
0  12573.0  461.466003         192.903351          84.572075      0.898772   
1  12845.0  464.121002         194.332214          85.524338      0.897952   
2  14055.0  488.748993         207.751755          87.250328      0.907536   
3  14412.0  490.324005         207.476135          89.689514      0.901735   
4  14658.0  477.117004         189.566635          99.997780      0.849551   
5  10578.0  414.619995         167.751923          81.585701      0.873766   
6  16122.0  539.000000         233.913513          89.531082      0.923851   
7  11119.0  427.045990         178.813126          80.617844      0.892600   
8  11075.0  416.850006         165.296265          86.279121      0.852966   
9  13066.0  458.261993         186.345856          90.901459      0.872950   

   Convex_Area    Extent   class  
0      12893.0  0.550433  class2  
1      13125.0  0.774962  class2 

In [6]:
# Pre-process dataset

# The class column holds these labels; they are encoded as the integers 0 and 1.
CLASS_LABELS = {'class1': 0, 'class2': 1}


def preprocess_dataset(data, class_labels=CLASS_LABELS):
    """Impute missing feature values with the column mean, scale the features to [0, 1]
    and encode the class column as integers.

    The number of features is read from the data, so any dataset with the class in the
    last column can be pre-processed with this function. The input DataFrame is not
    modified; new arrays are returned.

    Arguments:
        data: pandas DataFrame whose last column holds the class labels
        class_labels: mapping from class label to integer
    Returns:
        X: numpy array of shape (n_examples, n_features), values scaled to [0, 1]
        y: numpy array of shape (n_examples) of integer class values
    """
    # Everything except the last column is a feature; anything non-numeric becomes NaN
    # so that it is imputed rather than breaking the scaler.
    features = data.iloc[:, :-1].apply(pd.to_numeric, errors='coerce')
    classes = data.iloc[:, -1].astype(str).str.strip()

    imputed_features = SimpleImputer(strategy='mean').fit_transform(features)
    scaled_features = MinMaxScaler().fit_transform(imputed_features)
    encoded_classes = classes.map(class_labels).to_numpy(dtype=int)

    return scaled_features, encoded_classes


X, y = preprocess_dataset(rice_data)
print("Missing values remaining: {}".format(int(pd.isna(X).sum())))

Missing values remaining: 0


In [7]:
# Print first ten rows of pre-processed dataset to 4 decimal places as per assignment spec
# A function is provided to assist

def print_data(X, y, n_rows=10):
    """Takes a numpy data array and target and prints the first ten rows.
    
    Arguments:
        X: numpy array of shape (n_examples, n_features)
        y: numpy array of shape (n_examples)
        n_rows: numpy of rows to print
    """
    for example_num in range(n_rows):
        for feature in X[example_num]:
            print("{:.4f}".format(feature), end=",")

        if example_num == len(X)-1:
            print(y[example_num],end="")
        else:
            print(y[example_num])
            


# Print the first ten rows of the pre-processed rice dataset
print_data(X, y)

0.4628,0.5406,0.5113,0.4803,0.7380,0.4699,0.1196,1
0.4900,0.5547,0.5266,0.5018,0.7319,0.4926,0.8030,1
0.6109,0.6847,0.6707,0.5409,0.8032,0.6253,0.1185,0
0.6466,0.6930,0.6677,0.5961,0.7601,0.6467,0.2669,0
0.6712,0.6233,0.4755,0.8293,0.3721,0.6803,0.4211,1
0.2634,0.2932,0.2414,0.4127,0.5521,0.2752,0.2825,1
0.8175,0.9501,0.9515,0.5925,0.9245,0.8162,0.0000,0
0.3174,0.3588,0.3601,0.3908,0.6921,0.3261,0.8510,1
0.3130,0.3050,0.2150,0.5189,0.3974,0.3159,0.4570,1
0.5120,0.5237,0.4409,0.6235,0.5460,0.5111,0.3155,1


In [24]:
#load the test dataset to test out your model
test_data = load_dataset('test-before.csv')
X_before, y_before = preprocess_dataset(test_data)
print("test-before: {} examples, {} features".format(X_before.shape[0], X_before.shape[1]))
print("LogR runnability check accuracy: {:.4f}".format(
    cross_val_score(clf_logr, X_before, y_before, cv=cvKFold).mean()))

test-before: 209 examples, 6 features
LogR runnability check accuracy: 0.6700


## **2. Build Classifiers**

- Part 1:  Logistic Regression, Naïve Bayes
- Part 2:  KNN, Decision Tree, Ada Boost, Gradient Boost, Random Forest, SVM

### Part 1: Cross-validation without parameter tuning

In [23]:
## Setting the 10 fold stratified cross-validation
cvKFold=StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

# The stratified folds from cvKFold should be provided to the classifiers


from sklearn.model_selection import train_test_split,GridSearchCV,cross_val_score
# Shared train/test split for part 2 classifiers
X_train, X_test, y_train,y_test = train_test_split(X,y,stratify=y,random_state=0)

# Shared grid search helper
def run_grid_search(clf,param_grid):
    gs =GridSearchCV(clf,param_grid,cv=cvKFold, scoring='accuracy',n_jobs=-1)
    gs.fit(X_train,y_train)
    return gs, gs.score(X_test,y_test)

In [21]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression

clf_logr = LogisticRegression(random_state=0)
logr_scores = cross_val_score(clf_logr,X,y, cv=cvKFold)

In [17]:
# Naïve Bayes
from sklearn.naive_bayes import GaussianNB
clf_nb = GaussianNB()
nb_scores = cross_val_score(clf_nb,X,y, cv=cvKFold)

### Part 1 Results


In [18]:
# Print results for each classifier in part 1 to 4 decimal places here:
print("LogR average cross-validation accuracy: {:.4f}".format(logr_scores.mean()))
print("NB average cross-validation accuracy: {:.4f} ".format(nb_scores.mean()))

LogR average cross-validation accuracy: 0.9386
NB average cross-validation accuracy: 0.9264 


### Part 2: Cross-validation with parameter tuning

In [ ]:
# KNN 
# parameters you may consider
k = [1, 3, 5, 7]
p = [1, 2]


In [ ]:
# Decision Tree 
# parameters you may consider
max_depth = [3, 5, 7, 10]
min_samples_split = [2, 5, 10]
min_samples_leaf = [1, 2, 4]

In [ ]:
# Ada Boost
# parameters you may consider
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]

In [ ]:
# Gradient Boost
# parameters you may consider
max_depth = [1, 3, 5, 7]
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]

In [ ]:
# Random Forest
# You should use RandomForestClassifier from sklearn.ensemble with information gain and max_features set to ‘sqrt’.
# parameters you may consider
n_estimators = [10, 30, 60, 100]
max_leaf_nodes = [6, 12]



In [ ]:
# SVM
# parameters you may consider
C = [0.01, 0.1, 1, 5]
gamma = [0.01, 0.1, 1, 10]
# optional
kernel = []


### Part 2: Results

In [ ]:
# Perform Grid Search with 10-fold stratified cross-validation (GridSearchCV in sklearn). 
# The stratified folds from cvKFold should be provided to GridSearchV

# This should include using train_test_split from sklearn.model_selection with stratification and random_state=0
# Print results for each classifier here. All the reported results should be printed to 4 decimal places except for the integers such as "k", "p", n_estimators" and "max_leaf_nodes".

# example printing:
print("KNN best k: ")
print("KNN best p: ")
print("KNN cross-validation accuracy: ")
print("KNN test set accuracy: ")

...

print("RF best n_estimators: ")
print("RF best max_leaf_nodes: ")
print("RF cross-validation accuracy: ")
print("RF test set accuracy: ")
print("RF test set macro average F1: ")
print("RF test set weighted average F1: ")

KNN best k: 
KNN best p: 
KNN cross-validation accuracy: 
KNN test set accuracy: 

RF best n_estimators: 
RF best max_leaf_nodes: 
RF cross-validation accuracy: 
RF test set accuracy: 
RF test set macro average F1: 
RF test set weighted average F1: 


### Test your code

In [ ]:
#load the test dataset to test out your model 


## **3. Reflection and Discussion**



## **AI Acknowledgement**